# FastSpeech 2 Text-to-Speech (English & Telugu) — End-to-End Colab Pro Pipeline

### Runtime Strategy (Save Compute Units & Never Lose Progress!)
1. **Sections 1 to 4 (Data Prep + Fast MFA Alignment + Feature Extraction):** Run on a **CPU (High-RAM)** runtime (`Runtime -> Change runtime type -> CPU`). Every completed stage backs up immediately to `/content/drive/MyDrive/FastSpeech2` so a Colab disconnect never loses progress.
2. **Sections 5 to 7 (Vocoder Gate + Model Training + Test Inference & Plots):** Switch to an **L4 GPU** runtime (`Runtime -> Change runtime type -> L4 GPU`). Section 5 restores the preprocessed features from Google Drive in seconds and links `checkpoints/`, `logs/`, and `Results/` to Drive.

## Section 1: Mount Google Drive, Clone Repo & Install Dependencies

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/FastSpeech2"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL = "https://github.com/SandeshAwate6055/FastSpeech2-English-Telugu.git"
REPO_DIR = "/content/FastSpeech2-English-Telugu"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!python -c "import nltk; nltk.download('averaged_perceptron_tagger'); nltk.download('averaged_perceptron_tagger_eng'); nltk.download('cmudict')"

if not os.path.exists("English/hifigan/generator_universal.pth.tar"):
    !git clone --depth 1 https://github.com/ming024/FastSpeech2.git /tmp/ming_fs2
    !unzip -q -o /tmp/ming_fs2/hifigan/generator_universal.pth.tar.zip -d English/hifigan/
    !unzip -q -o /tmp/ming_fs2/hifigan/generator_universal.pth.tar.zip -d Telugu/hifigan/
    print("Extracted generator_universal.pth.tar into English/hifigan and Telugu/hifigan")


## Section 2: Install Montreal Forced Aligner (MFA 2.2.17)

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
if not os.path.exists("/usr/local/bin/micromamba"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
    !mv -f bin/micromamba /usr/local/bin/micromamba

if not os.path.exists("/root/.local/share/mamba/envs/mfa"):
    !/usr/local/bin/micromamba create -y -n mfa -c conda-forge montreal-forced-aligner=2.2.17 openfst=1.8.2 kaldi=5.5.1068 "joblib<1.4" "setuptools<81" python=3.10

!/usr/local/bin/micromamba run -n mfa mfa version


## Section 3: English Pipeline (Data Prep -> MFA Alignment -> `preprocess.py` -> Immediate Drive Backup)
- Automatically skips if `English_preprocessed.tar.gz` is already saved on your Google Drive!

In [ ]:
import os
from getpass import getpass
os.environ["MPLBACKEND"] = "Agg"
DRIVE_ROOT = "/content/drive/MyDrive/FastSpeech2"
en_archive = f"{DRIVE_ROOT}/English_preprocessed.tar.gz"

if os.path.exists(en_archive):
    print(f"FOUND existing backup at {en_archive}! Restoring instead of re-running ...")
    !tar -xzf {en_archive} -C /content/FastSpeech2-English-Telugu/English
else:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        if not os.environ.get("HF_TOKEN"):
            os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face Read Token (hf_...): ")

    %cd /content/FastSpeech2-English-Telugu/English
    !python prepare_data.py --train_hours 5.0 --val_hours 1.0

    !/usr/local/bin/micromamba run -n mfa mfa model download acoustic english_us_arpa
    !/usr/local/bin/micromamba run -n mfa mfa align /content/FastSpeech2-English-Telugu/English/raw_data /content/FastSpeech2-English-Telugu/English/lexicon/english_svarah_mfa.dict english_us_arpa /content/FastSpeech2-English-Telugu/English/preprocessed_data/TextGrid --clean --single_speaker --num_jobs 4

    !python preprocess.py --config config.yaml
    !tar -czf {en_archive} -C /content/FastSpeech2-English-Telugu/English preprocessed_data lexicon raw_data/data_stats.json raw_data/train_filelist.txt raw_data/val_filelist.txt
    print(f"SUCCESS: Backed up English preprocessed features to {en_archive}!")


## Section 4: Telugu Pipeline (Data Prep -> Fast MFA Train & Align -> `preprocess.py` -> Immediate Drive Backup)
- Uses a streamlined MFA acoustic config (`monophone` + `triphone` + `lda` with `--single_speaker --num_jobs 4`) so Telugu alignment finishes in **~5 minutes** instead of 40 minutes!
- Automatically backs up `Telugu_preprocessed.tar.gz` to Google Drive as soon as it finishes.

In [ ]:
import os, shutil
os.environ["MPLBACKEND"] = "Agg"
DRIVE_ROOT = "/content/drive/MyDrive/FastSpeech2"
te_archive = f"{DRIVE_ROOT}/Telugu_preprocessed.tar.gz"

if os.path.exists(te_archive):
    print(f"FOUND existing backup at {te_archive}! Restoring instead of re-running ...")
    !tar -xzf {te_archive} -C /content/FastSpeech2-English-Telugu/Telugu
else:
    %cd /content/FastSpeech2-English-Telugu/Telugu
    !python prepare_data.py --train_hours 5.0 --val_hours 1.0

    # Build train-only symlink directory for Telugu MFA acoustic training (no validation leakage)
    te_raw = "/content/FastSpeech2-English-Telugu/Telugu/raw_data"
    te_train_only = "/content/te_mfa_train_corpus"
    if os.path.exists(te_train_only):
        shutil.rmtree(te_train_only)
    for mf in ["train_filelist.txt", "reserve_train_filelist.txt"]:
        p = os.path.join(te_raw, mf)
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        utt_id, spk_id = line.strip().split("|")[:2]
                        os.makedirs(os.path.join(te_train_only, spk_id), exist_ok=True)
                        for ext in [".wav", ".lab"]:
                            src = os.path.join(te_raw, spk_id, utt_id + ext)
                            dst = os.path.join(te_train_only, spk_id, utt_id + ext)
                            if os.path.exists(src) and not os.path.exists(dst):
                                os.symlink(src, dst)

    # Streamlined MFA config (monophone + triphone + lda; skips slow 3x SAT passes) -> finishes in ~5 mins!
    fast_mfa_cfg = """beam: 10
retry_beam: 40
features:
  type: mfcc
  use_energy: false
  frame_shift: 10
training:
  - monophone:
      num_iterations: 20
      max_gaussians: 1000
  - triphone:
      num_iterations: 20
      num_leaves: 2000
      max_gaussians: 10000
  - lda:
      num_iterations: 20
      num_leaves: 2500
      max_gaussians: 15000
"""
    with open("/content/fast_mfa_train.yaml", "w") as f:
        f.write(fast_mfa_cfg)

    !/usr/local/bin/micromamba run -n mfa mfa train /content/te_mfa_train_corpus /content/FastSpeech2-English-Telugu/Telugu/lexicon/telugu-lexicon.txt /content/FastSpeech2-English-Telugu/Telugu/te_mfa_acoustic.zip --config_path /content/fast_mfa_train.yaml --single_speaker --clean --num_jobs 4
    !/usr/local/bin/micromamba run -n mfa mfa align /content/FastSpeech2-English-Telugu/Telugu/raw_data /content/FastSpeech2-English-Telugu/Telugu/lexicon/telugu-lexicon.txt /content/FastSpeech2-English-Telugu/Telugu/te_mfa_acoustic.zip /content/FastSpeech2-English-Telugu/Telugu/preprocessed_data/TextGrid --single_speaker --clean --num_jobs 4

    !python preprocess.py --config config.yaml
    !tar -czf {te_archive} -C /content/FastSpeech2-English-Telugu/Telugu preprocessed_data lexicon raw_data/data_stats.json raw_data/train_filelist.txt raw_data/val_filelist.txt
    print(f"SUCCESS: Backed up Telugu preprocessed features to {te_archive}!")


## Section 5: Restore Preprocessed Data on GPU Runtime & Verify Vocoder Gate
When you switch to an **L4 GPU** runtime, run **Section 1** and then jump directly here to **Section 5**!

In [ ]:
import os, shutil

DRIVE_ROOT = "/content/drive/MyDrive/FastSpeech2"
REPO_DIR = "/content/FastSpeech2-English-Telugu"
%cd {REPO_DIR}

if not os.path.exists("English/preprocessed_data/train.txt") and os.path.exists(f"{DRIVE_ROOT}/English_preprocessed.tar.gz"):
    !tar -xzf {DRIVE_ROOT}/English_preprocessed.tar.gz -C {REPO_DIR}/English
    print("Restored English preprocessed_data from Drive.")

if not os.path.exists("Telugu/preprocessed_data/train.txt") and os.path.exists(f"{DRIVE_ROOT}/Telugu_preprocessed.tar.gz"):
    !tar -xzf {DRIVE_ROOT}/Telugu_preprocessed.tar.gz -C {REPO_DIR}/Telugu
    print("Restored Telugu preprocessed_data from Drive.")

# Link checkpoints, logs, and Results to Google Drive for real-time persistence
for lang in ["English", "Telugu"]:
    for sub in ["checkpoints", "logs"]:
        drive_sub = os.path.join(DRIVE_ROOT, lang, sub)
        os.makedirs(drive_sub, exist_ok=True)
        local_sub = os.path.join(REPO_DIR, lang, sub)
        if os.path.islink(local_sub):
            os.unlink(local_sub)
        elif os.path.isdir(local_sub):
            shutil.rmtree(local_sub)
        os.symlink(drive_sub, local_sub)

drive_results = os.path.join(DRIVE_ROOT, "Results")
os.makedirs(drive_results, exist_ok=True)
local_results = os.path.join(REPO_DIR, "Results")
if os.path.islink(local_results):
    os.unlink(local_results)
elif os.path.isdir(local_results):
    shutil.rmtree(local_results)
os.symlink(drive_results, local_results)
print("Linked checkpoints, logs, and Results to Google Drive!")


In [ ]:
# Vocoder Gate: Reconstruct 1 ground-truth Mel-spectrogram per language through Universal HiFi-GAN
import sys, glob, os, yaml, torch, numpy as np
from IPython.display import Audio, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Active device:", device)

for lang in ["English", "Telugu"]:
    lang_dir = f"/content/FastSpeech2-English-Telugu/{lang}"
    sys.path.insert(0, lang_dir)
    from utils.model import get_vocoder, vocoder_infer
    cfg = yaml.safe_load(open(f"{lang_dir}/config.yaml"))
    vocoder = get_vocoder(cfg, device)
    mel_files = sorted(glob.glob(f"{lang_dir}/preprocessed_data/mel/*.npy"))
    if mel_files:
        mel = np.load(mel_files[0])
        mel_tensor = torch.from_numpy(mel).float().unsqueeze(0).transpose(1, 2).to(device)
        wav = vocoder_infer(mel_tensor, vocoder, cfg, cfg)[0]
        print(f"=== {lang} Ground-Truth Mel Reconstruction ({os.path.basename(mel_files[0])}) ===")
        display(Audio(wav, rate=22050))
    sys.path.pop(0)


## Section 6: Train English & Telugu FastSpeech 2 Models (`train.py`)
- Automatically detects and resumes from the latest checkpoint in Google Drive if interrupted.

In [ ]:
# 6A. Train English FastSpeech 2 Model
import glob, os
%cd /content/FastSpeech2-English-Telugu/English
ckpts = [int(os.path.basename(f).replace(".pth.tar", "")) for f in glob.glob("checkpoints/*.pth.tar") if os.path.basename(f).replace(".pth.tar", "").isdigit()]
restore_step = max(ckpts) if ckpts else 0
print(f"Starting/Resuming English training from step {restore_step} ...")
!python train.py --config config.yaml --restore_step {restore_step}


In [ ]:
# 6B. Train Telugu FastSpeech 2 Model
import glob, os
%cd /content/FastSpeech2-English-Telugu/Telugu
ckpts = [int(os.path.basename(f).replace(".pth.tar", "")) for f in glob.glob("checkpoints/*.pth.tar") if os.path.basename(f).replace(".pth.tar", "").isdigit()]
restore_step = max(ckpts) if ckpts else 0
print(f"Starting/Resuming Telugu training from step {restore_step} ...")
!python train.py --config config.yaml --restore_step {restore_step}


## Section 7: Synthesize All 10 Test Sentences & Generate Variance Adaptor Plots (`inference.py`)

In [ ]:
%cd /content/FastSpeech2-English-Telugu/English
!python inference.py --config config.yaml --speaker_id 0

%cd /content/FastSpeech2-English-Telugu/Telugu
!python inference.py --config config.yaml --speaker_id 0

%cd /content/FastSpeech2-English-Telugu
!zip -r /content/drive/MyDrive/FastSpeech2/Results_Deliverables.zip Results/
print("All 10 test audios, 30 variance adaptor plots, and 4 loss curves saved to Google Drive!")
